Import Required Libraries

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/02_dimension_data_processing/utilities

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','orders','Data Source')

catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

base_path=f"/Volumes/{catalog}/{bronze_schema}/sports_volume/{data_source}"
landing_path=f'{base_path}/landing/'
processed_path=f'{base_path}/processed/'

bronze_table=f"{catalog}.{bronze_schema}.{data_source}"
silver_table=f"{catalog}.{silver_schema}.{data_source}"
gold_table=f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df=spark.read.options(header=True,inferSchema=True).csv(f'{landing_path}/*.csv').withColumn('read_time_stamp',current_timestamp())
df.count()
display(df.limit(10))

In [0]:
df.count()


In [0]:
df.write.format('delta')\
    .option('delta.enableChangeDataFeed','true')\
    .mode('append')\
    .saveAsTable(bronze_table)

In [0]:
# creating staging table for the incremental data
df.write.format('delta')\
    .option('delta.enableChangeDataFeed','true')\
    .mode('overwrite')\
    .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

moving files from source to processed directory

In [0]:
files=dbutils.fs.ls(landing_path)
for file in files:
    dbutils.fs.mv(file.path,f"{processed_path}/{file.name}",True)

Silver

In [0]:
df_silver=spark.sql(f'select * from {catalog}.{bronze_schema}.staging_{data_source}')
display(df_silver)


In [0]:
df_silver=df_silver.filter(col('order_qty').isNotNull())
df_silver=df_silver.withColumn('customer_id',
                               when(
                                   col('customer_id').rlike("^[0-9]+$"),col('customer_id')
                               ).otherwise(999999).cast('string')
                               )
df_silver=df_silver.withColumn('order_placement_date',regexp_replace('order_placement_date',r'^[A-Za-z]+,\s*',""))

df_silver=df_silver.withColumn('order_placement_date', coalesce(
try_to_date('order_placement_date','yyyy/MM/dd'),
try_to_date('order_placement_date','dd-MM-yyyy'),
 try_to_date('order_placement_date','dd/MM/yyyy'),
  try_to_date('order_placement_date','MMMM dd, yyyy')                                                                ))

df_silver=df_silver.dropDuplicates(['order_id','order_placement_date','customer_id','product_id','order_qty'])
df_silver=df_silver.withColumn('product_id', col('product_id').cast('string'))
df_silver.agg(min("order_placement_date").alias('min_date'),max("order_placement_date").alias('max_date')).show()  


In [0]:
display(df_silver)

In [0]:
df_silver.agg(min("order_placement_date").alias('min_date'),max("order_placement_date").alias('max_date')).show()

In [0]:
df_products=spark.table('fmcg.silver.products')

In [0]:
df_joined=df_silver.join(df_products,on="product_id",how='inner').select(df_silver["*"],df_products['product_code'])
display(df_joined.limit(10))

In [0]:
df_joined.count()


In [0]:
if not  (spark.catalog.tableExists(silver_table)):
    df_joined.write.format('delta').option('delta.enableChangeDataFeed','true').option('mergeSchema','true').mode('overwrite').saveAsTable(silver_table)
else:
    silver_delta=DeltaTable.forName(spark,silver_table)
    silver_delta.alias('silver').merge(
        source=df_joined.alias('source'),
        condition='''
        silver.order_placement_date=source.order_placement_date
         and silver.order_id=source.order_id 
         and silver.product_code=source.product_code 
         and silver.customer_id=source.customer_id
        '''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

Staging table to process just the arrived ncremental data



In [0]:
# creating staging table for the incremental data
df_joined.write.format('delta')\
    .option('delta.enableChangeDataFeed','true')\
    .mode('overwrite')\
    .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

In [0]:
df_gold=spark.sql(f"select order_id, order_placement_date as date, customer_id as customer_code,product_code, product_id,order_qty as sold_quantity from {catalog}.{silver_schema}.staging_{data_source}")
display(df_gold.limit(20))

In [0]:
gold_table

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    df_gold.write.format('delta').option('mergeSchema','true').option('delta.enableChangeDataFeed','true').mode('overwrite').saveAsTable(gold_table)
else:
    delta_table=DeltaTable.forName(spark,gold_table)
    delta_table.alias('target').merge(
        source=df_gold.alias('source'),
        condition='''source.date=target.date
        and source.order_id=target.order_id
        and source.product_code=target.product_code
        '''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

merging with parent company

In [0]:
df_child=spark.sql(f"select order_placement_date as date from {catalog}.{silver_schema}.staging_{data_source}")
incremental_month_df=df_child.select(trunc('date','MM').alias('start_month')).distinct()
incremental_month_df.show()
incremental_month_df.createOrReplaceTempView('incremental_months')



In [0]:
monthly_table=spark.sql(f"""
                        select date, product_code,customer_code,sold_quantity from {catalog}.{gold_schema}.sb_fact_orders a 
                        inner join incremental_months b
                        on trunc(a.date,"MM")=b.start_month
                        """)
monthly_table.show()

In [0]:
monthly_table.count()

In [0]:
monthly_table_gold=monthly_table.withColumn('date', trunc('date','MM')).groupBy("date",'product_code','customer_code').agg(sum("sold_quantity").alias('sold_quantity'))
display(monthly_table_gold)

In [0]:
gold_parent=DeltaTable.forName(spark,f"{catalog}.{gold_schema}.fact_orders")
gold_parent.alias('target').merge(
    source=monthly_table_gold.alias('source'),
    condition='''
    source.date=target.date
    and source.product_code=target.product_code
    and source.customer_code=target.customer_code
    '''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{bronze_schema}.staging_{data_source}")
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{silver_schema}.staging_{data_source}")

In [0]:
tables=spark.sql(f"select * from fmcg.gold.fact_orders")
tables.count()

In [0]:
from pyspark.sql.functions import max

tables.agg(max("date").alias("max_date")).show()

In [0]:
tables.select('date') \
    .filter(col('date').between('2025-12-01', '2025-12-21')) \
    .show()